In [30]:
import math
import random
from collections import defaultdict ,  Counter



In [31]:


class Markov :
    def __init__(self, corpus , vocab_size = None):
        self.corpus = corpus
        self.words = corpus.split()
        self.v = vocab_size if vocab_size is not None else len(set(self.words))
        self.trans = self._build_trans()
        self.totals = {}
        for c , n in self.trans.items():
            self.totals[c] = sum(n.values())
        self._probs = None
    


    def _build_trans(self):
        trans = defaultdict(Counter)
        words = self.words

        for i in range(len(words) - 1):
            trans[words[i]][words[i + 1]] += 1

        return trans


    @property
    def probs(self):
        if self._probs is None:
            self._probs = {
                cur: {w: c / self.totals[cur] for w, c in nxt.items()}
                for cur, nxt in self.trans.items()
            }
        return self._probs

    def generate(self, start="the", length=30):
        probs = self.probs
        current = start
        sentence = [current]
        for _ in range(length):
            if current not in probs:
                break
            current = random.choices(
                list(probs[current].keys()), weights=probs[current].values()
            )[0]
            sentence.append(current)
        return " ".join(sentence)

    def check_sentence(self, sentence):
        trans = self.trans      
        totals = self.totals    
        v = self.v              
        words = sentence.split()
        log_prob = 0.0
        for i in range(len(words) - 1):
            current, next_word = words[i], words[i + 1]
            total = totals.get(current, 0)
            count = trans[current][next_word] if current in trans else 0
            p = (count + 1) / (total + v)
            log_prob += math.log(p)
        return log_prob / (len(words) - 1)

In [32]:

import re

SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")
 
 
def split_into_sentences(text, min_words=4):
    text = re.sub(r"\s+", " ", text.strip())
    raw = SENTENCE_SPLIT_RE.split(text)
    return [s.strip() for s in raw if len(s.split()) >= min_words]
 
 
def train_test_split_sentences(sentences, test_ratio=0.2, seed=42):
    sentences = sentences[:]
    random.Random(seed).shuffle(sentences)
    cut = int(len(sentences) * (1 - test_ratio))
    return sentences[:cut], sentences[cut:]
 

In [33]:
def classify(sentence, model_a, model_b):
    return "a" if model_a.check_sentence(sentence) > model_b.check_sentence(sentence) else "b"
 
 
def perplexity(model, sentence):
    return math.exp(-model.check_sentence(sentence))
 
 

In [34]:

def evaluate(model_a, model_b, test_a, test_b, label_a="A", label_b="B"):
    tp = fn = fp = tn = 0
    for s in test_a:
        if classify(s, model_a, model_b) == "a":
            tp += 1
        else:
            fn += 1
    for s in test_b:
        if classify(s, model_a, model_b) == "b":
            tn += 1
        else:
            fp += 1
 
    total = tp + fp + fn + tn
    accuracy = (tp + tn) / total if total else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
 
    print("Confusion matrix (rows = actual, cols = predicted):")
    print(f"{'':>20}{'pred ' + label_a:>18}{'pred ' + label_b:>18}")
    print(f"{'actual ' + label_a:>20}{tp:>18}{fn:>18}")
    print(f"{'actual ' + label_b:>20}{fp:>18}{tn:>18}")
    print()
    print(f"Accuracy:            {accuracy:.3f}")
    print(f"Precision ({label_a}):     {precision:.3f}")
    print(f"Recall ({label_a}):        {recall:.3f}")
    print(f"F1 ({label_a}):            {f1:.3f}")
 
    own_a = sum(perplexity(model_a, s) for s in test_a) / len(test_a)
    cross_a = sum(perplexity(model_b, s) for s in test_a) / len(test_a)
    own_b = sum(perplexity(model_b, s) for s in test_b) / len(test_b)
    cross_b = sum(perplexity(model_a, s) for s in test_b) / len(test_b)
 
    print()
    print("Perplexity (lower = better fit; own model should beat the other):")
    print(f"  {label_a} test text -> {label_a} model: {own_a:8.1f}   {label_b} model: {cross_a:8.1f}")
    print(f"  {label_b} test text -> {label_b} model: {own_b:8.1f}   {label_a} model: {cross_b:8.1f}")
 
    return {
        "accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1,
        "confusion": {"tp": tp, "fp": fp, "fn": fn, "tn": tn},
        "perplexity": {"own_a": own_a, "cross_a": cross_a, "own_b": own_b, "cross_b": cross_b},
    }
 
 

 

In [ ]:
with open("nieText.txt", encoding="utf-8") as f:
    nietzsche_text = f.read()
with open("schText.txt", encoding="utf-8") as f:
    schopenhauer_text = f.read()

niet_sentences = split_into_sentences(nietzsche_text)
sch_sentences = split_into_sentences(schopenhauer_text)
niet_train, niet_test = train_test_split_sentences(niet_sentences, test_ratio=0.3)
sch_train, sch_test = train_test_split_sentences(sch_sentences, test_ratio=0.3)
print(f"Nietzsche: {len(niet_train)} train sentences, {len(niet_test)} test sentences")
print(f"Schopenhauer: {len(sch_train)} train sentences, {len(sch_test)} test sentences")
print()
shared_vocab = set(" ".join(niet_train).split()) | set(" ".join(sch_train).split())
niet_model = Markov(" ".join(niet_train), vocab_size=len(shared_vocab))
sch_model = Markov(" ".join(sch_train), vocab_size=len(shared_vocab))
evaluate(niet_model, sch_model, niet_test, sch_test, label_a="Nietzsche", label_b="Schopenhauer")
print()
print(niet_model.generate(start="life"))
print(sch_model.generate(start="life"))

Nietzsche: 6979 train sentences, 2992 test sentences
Schopenhauer: 1630 train sentences, 699 test sentences

Confusion matrix (rows = actual, cols = predicted):
                        pred Nietzsche pred Schopenhauer
    actual Nietzsche              2900                92
 actual Schopenhauer               598               101

Accuracy:            0.813
Precision (Nietzsche):     0.829
Recall (Nietzsche):        0.969
F1 (Nietzsche):            0.894

Perplexity (lower = better fit; own model should beat the other):
  Nietzsche test text -> Nietzsche model:  13211.3   Schopenhauer model:  21514.0
  Schopenhauer test text -> Schopenhauer model:  13507.9   Nietzsche model:  11638.4

life and height, and reverent multitude, above it would find themselves write it--would almost with our food of taking place is. Or, to DISCHARGE its worst, the existence of obedience is
life it is why this file produced the processes of gravity is necessary and surmounted with the tribunal which so Natur

Nietzsche: 6979 train sentences (raw), 2992 test sentences
Schopenhauer: 1630 train sentences (raw), 699 test sentences
After balancing: 1630 train sentences each
Avg words/sentence -> Nietzsche: 26.3, Schopenhauer: 33.5
Total training words -> Nietzsche: 42912, Schopenhauer: 54671

Confusion matrix (rows = actual, cols = predicted):
                        pred Nietzsche pred Schopenhauer
    actual Nietzsche              1638              1354
 actual Schopenhauer                29               670

Accuracy:                0.625
Majority-class baseline: 0.811  (always guessing the bigger test class)

Precision (Nietzsche):         0.983
Recall (Nietzsche):            0.547
F1 (Nietzsche):                0.703

Precision (Schopenhauer):      0.331
Recall (Schopenhauer):         0.959
F1 (Schopenhauer):             0.492

Balanced accuracy:       0.753  (avg of per-class recall - fair under class imbalance)
Macro-F1:                0.598  (avg of per-class F1)

Perplexity (lower = be